In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import pickle


In [9]:
df = pd.read_csv('game_of_thrones_train.csv', index_col='S.No')
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 1557 entries, 1 to 1557
Data columns (total 25 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   name              1557 non-null   object 
 1   title             717 non-null    object 
 2   male              1557 non-null   int64  
 3   culture           488 non-null    object 
 4   dateOfBirth       279 non-null    float64
 5   mother            18 non-null     object 
 6   father            22 non-null     object 
 7   heir              21 non-null     object 
 8   house             1176 non-null   object 
 9   spouse            200 non-null    object 
 10  book1             1557 non-null   int64  
 11  book2             1557 non-null   int64  
 12  book3             1557 non-null   int64  
 13  book4             1557 non-null   int64  
 14  book5             1557 non-null   int64  
 15  isAliveMother     18 non-null     float64
 16  isAliveFather     22 non-null     float64
 17  

,name,title,male,culture,dateOfBirth,mother,father,heir,house,spouse,...,isAliveMother,isAliveFather,isAliveHeir,isAliveSpouse,isMarried,isNoble,age,numDeadRelations,popularity,isAlive
S.No,,,,,,,,,,,,,,,,,,,,,
1,Viserys II Targaryen,NaN,1,NaN,NaN,Rhaenyra Targaryen,Daemon Targaryen,Aegon IV Targaryen,NaN,NaN,...,1.0,0.0,0.0,NaN,0,0,NaN,11,0.605351,0
2,Walder Frey,Lord of the Crossing,1,Rivermen,208.0,NaN,NaN,NaN,House Frey,Perra Royce,...,NaN,NaN,NaN,1.0,1,1,97.0,1,0.896321,1
3,Addison Hill,Ser,1,NaN,NaN,NaN,NaN,NaN,House Swyft,NaN,...,NaN,NaN,NaN,NaN,0,1,NaN,0,0.267559,1
4,Aemma Arryn,Queen,0,NaN,82.0,NaN,NaN,NaN,House Arryn,Viserys I Targaryen,...,NaN,NaN,NaN,0.0,1,1,23.0,0,0.183946,0
5,Sylva Santagar,Greenstone,0,Dornish,276.0,NaN,NaN,NaN,House Santagar,Eldon Estermont,...,NaN,NaN,NaN,1.0,1,1,29.0,0,0.043478,1


In [10]:
df.dropna(axis=1, thresh=50, inplace=True)


In [11]:
df['popularity_log'] = np.log10(df['popularity'] * 10 + 1)
df.drop(columns=['popularity'], inplace=True)


In [12]:
df['boolDeadRelations'] = (df['numDeadRelations'] > 0).astype(int)
df.drop(columns=['numDeadRelations'], inplace=True)
df.head()

,name,title,male,culture,dateOfBirth,house,spouse,book1,book2,book3,book4,book5,isAliveSpouse,isMarried,isNoble,age,isAlive,popularity_log,boolDeadRelations
S.No,,,,,,,,,,,,,,,,,,,
1,Viserys II Targaryen,NaN,1,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,0,0,NaN,0,0.848405,1
2,Walder Frey,Lord of the Crossing,1,Rivermen,208.0,House Frey,Perra Royce,1,1,1,1,1,1.0,1,1,97.0,1,0.998399,1
3,Addison Hill,Ser,1,NaN,NaN,House Swyft,NaN,0,0,0,1,0,NaN,0,1,NaN,1,0.565327,0
4,Aemma Arryn,Queen,0,NaN,82.0,House Arryn,Viserys I Targaryen,0,0,0,0,0,0.0,1,1,23.0,0,0.453237,0
5,Sylva Santagar,Greenstone,0,Dornish,276.0,House Santagar,Eldon Estermont,0,0,0,1,0,1.0,1,1,29.0,1,0.156786,0


In [13]:
df.drop(columns=['dateOfBirth'], inplace=True)

In [14]:
df['age_no_data'] = df['age'].isna().astype(int)
df['age_value'] = df['age'].fillna(0)
df.drop(columns=['age'], inplace=True)
df.head()

,name,title,male,culture,house,spouse,book1,book2,book3,book4,book5,isAliveSpouse,isMarried,isNoble,isAlive,popularity_log,boolDeadRelations,age_no_data,age_value
S.No,,,,,,,,,,,,,,,,,,,
1,Viserys II Targaryen,NaN,1,NaN,NaN,NaN,0,0,0,0,0,NaN,0,0,0,0.848405,1,1,0.0
2,Walder Frey,Lord of the Crossing,1,Rivermen,House Frey,Perra Royce,1,1,1,1,1,1.0,1,1,1,0.998399,1,0,97.0
3,Addison Hill,Ser,1,NaN,House Swyft,NaN,0,0,0,1,0,NaN,0,1,1,0.565327,0,1,0.0
4,Aemma Arryn,Queen,0,NaN,House Arryn,Viserys I Targaryen,0,0,0,0,0,0.0,1,1,0,0.453237,0,0,23.0
5,Sylva Santagar,Greenstone,0,Dornish,House Santagar,Eldon Estermont,0,0,0,1,0,1.0,1,1,1,0.156786,0,0,29.0


In [16]:
cultures_grouped = {
    'Old Nations': ['valyrian', 'first men', 'andal', 'andals', 'rhoynar'],
    'the North': ['northmen', 'northern mountain clans', 'crannogmen'],
    'the Iron Islands': ['ironborn', 'ironborn', 'ironmen'],
    'the Mountain and the Vale': ['valemen', 'vale', 'vale mountain clans', 'sistermen'],
    'the Isles and Rivers': ['riverlands', 'rivermen'],
    'the Rock': ['westerman', 'westermen', 'westerlands'],
    'the Stormlands': ['stormlander', 'stormlands'],
    'the Reach': ['reach', 'reachmen', 'the reach'],
    'Dorne': ['dornish', 'dornishmen', 'dorne'],
    'Essos Nations': ['astapor', 'astapori', 'braavosi', 'braavos', 'tyroshi', 'lysene', 'lyseni',
                      'myrish', 'pentoshi', 'qartheen', 'qarth', 'dothraki',
                      'lhazarene', 'lhazareen','meereen', 'meereenese',
                      'norvoshi', 'qohor', 'summer isles', 'summer islands',
                      'summer islander', 'asshai', "asshai'i", 'norvos', 'ghiscari',
                      'ghiscaricari'],
    'Other Nations': ['ibbenese', 'westeros', 'free folk', 'wildling', 'wildlings', 'naathi']
}

In [17]:
culture_to_group = {}
for group, values in cultures_grouped.items():
    for v in values:
        culture_to_group[v] = group
df['culture'] = df['culture'].str.lower().str.strip().replace(culture_to_group)
df['culture'] = df['culture'].fillna('culture_no_data')

In [18]:
title_counts = df['title'].value_counts()
rare_titles = title_counts[title_counts < 3].index.tolist()
df['title'] = df['title'].replace(rare_titles, 'Rare')
df['title'] = df['title'].fillna('unknown_title')
df.head()

,name,title,male,culture,house,spouse,book1,book2,book3,book4,book5,isAliveSpouse,isMarried,isNoble,isAlive,popularity_log,boolDeadRelations,age_no_data,age_value
S.No,,,,,,,,,,,,,,,,,,,
1,Viserys II Targaryen,unknown_title,1,culture_no_data,NaN,NaN,0,0,0,0,0,NaN,0,0,0,0.848405,1,1,0.0
2,Walder Frey,Rare,1,the Isles and Rivers,House Frey,Perra Royce,1,1,1,1,1,1.0,1,1,1,0.998399,1,0,97.0
3,Addison Hill,Ser,1,culture_no_data,House Swyft,NaN,0,0,0,1,0,NaN,0,1,1,0.565327,0,1,0.0
4,Aemma Arryn,Queen,0,culture_no_data,House Arryn,Viserys I Targaryen,0,0,0,0,0,0.0,1,1,0,0.453237,0,0,23.0
5,Sylva Santagar,Rare,0,Dorne,House Santagar,Eldon Estermont,0,0,0,1,0,1.0,1,1,1,0.156786,0,0,29.0


In [19]:
house_counts = df['house'].value_counts()
rare_houses = house_counts[house_counts < 3].index.tolist()
df['house'] = df['house'].replace(rare_houses, 'Rare_house')
df['house'] = df['house'].fillna('house_unknown')


In [21]:
def spouse_status(row):
    if pd.isna(row['isAliveSpouse']):
        return 'no_spouse'
    elif row['isAliveSpouse'] == 1:
        return 'spouse_alive'
    else:
        return 'spouse_dead'

In [22]:
df['spouse_status'] = df.apply(spouse_status, axis=1)
spouse_dummies = pd.get_dummies(df['spouse_status'], prefix='spouse', dtype=int)
df = pd.concat([df, spouse_dummies], axis=1)
df.drop(columns=['isAliveSpouse', 'spouse_status'], inplace=True)
df.head()

,name,title,male,culture,house,spouse,book1,book2,book3,book4,...,isMarried,isNoble,isAlive,popularity_log,boolDeadRelations,age_no_data,age_value,spouse_no_spouse,spouse_spouse_alive,spouse_spouse_dead
S.No,,,,,,,,,,,,,,,,,,,,,
1,Viserys II Targaryen,unknown_title,1,culture_no_data,house_unknown,NaN,0,0,0,0,...,0,0,0,0.848405,1,1,0.0,1,0,0
2,Walder Frey,Rare,1,the Isles and Rivers,House Frey,Perra Royce,1,1,1,1,...,1,1,1,0.998399,1,0,97.0,0,1,0
3,Addison Hill,Ser,1,culture_no_data,House Swyft,NaN,0,0,0,1,...,0,1,1,0.565327,0,1,0.0,1,0,0
4,Aemma Arryn,Queen,0,culture_no_data,House Arryn,Viserys I Targaryen,0,0,0,0,...,1,1,0,0.453237,0,0,23.0,0,0,1
5,Sylva Santagar,Rare,0,Dorne,Rare_house,Eldon Estermont,0,0,0,1,...,1,1,1,0.156786,0,0,29.0,0,1,0


In [23]:
df = pd.concat([df, pd.get_dummies(df['title'], prefix='title')], axis=1)
df = pd.concat([df, pd.get_dummies(df['culture'], prefix='culture')], axis=1)
df = pd.concat([df, pd.get_dummies(df['house'], prefix='house')], axis=1)
df.head()

,name,title,male,culture,house,spouse,book1,book2,book3,book4,...,house_House Whent,house_House Wylde,house_House of Galare,house_House of Loraq,house_Kingsguard,house_Night's Watch,house_Rare_house,house_Second Sons,house_Stone Crows,house_house_unknown
S.No,,,,,,,,,,,,,,,,,,,,,
1,Viserys II Targaryen,unknown_title,1,culture_no_data,house_unknown,NaN,0,0,0,0,...,False,False,False,False,False,False,False,False,False,True
2,Walder Frey,Rare,1,the Isles and Rivers,House Frey,Perra Royce,1,1,1,1,...,False,False,False,False,False,False,False,False,False,False
3,Addison Hill,Ser,1,culture_no_data,House Swyft,NaN,0,0,0,1,...,False,False,False,False,False,False,False,False,False,False
4,Aemma Arryn,Queen,0,culture_no_data,House Arryn,Viserys I Targaryen,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
5,Sylva Santagar,Rare,0,Dorne,Rare_house,Eldon Estermont,0,0,0,1,...,False,False,False,False,False,False,True,False,False,False


In [24]:
df.drop(columns=['title', 'culture', 'house', 'name', 'spouse'], inplace=True, errors='ignore')

In [25]:
y = df['isAlive']
X = df.drop(columns=['isAlive'])
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


In [26]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_val = model.predict(X_val)

acc_train = accuracy_score(y_train, y_pred_train)
acc_val = accuracy_score(y_val, y_pred_val)
print(f'Accuracy на обучении: {acc_train:.4f}')
print(f'Accuracy на валидации: {acc_val:.4f}')

Accuracy на обучении: 0.8426
Accuracy на валидации: 0.7917


In [27]:
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)
with open('train_columns.pkl', 'wb') as f:
    pickle.dump(X_train.columns.tolist(), f)

print("Модель и список столбцов сохранены.")

Модель и список столбцов сохранены.
